[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/28_moe.ipynb)

# 🔴 Hard: Mixture of Experts (MoE)

Implement a **Mixture of Experts** layer (Mixtral / Switch Transformer style).

### Signature
```python
class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, S, D) -> (B, S, D)
```

### Architecture
- `self.router`: `nn.Linear(d_model, num_experts)` — gating network
- `self.experts`: `nn.ModuleList` of MLPs `(Linear→ReLU→Linear)`
- For each token: select top-k experts, compute weighted sum of their outputs

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [2]:
import torch
import torch.nn as nn

/home/user/miniconda/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [30]:
# ✏️ YOUR IMPLEMENTATION HERE

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()
        # pass  # router + experts
        self.top_k = top_k
        self.router = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model)) for _ in range(num_experts)   
        ])

    def forward(self, x):
        D = x.size(-1)
        flat = x.view(-1, D)
        logits = self.router(flat)
        top_vals, top_idx = logits.topk(self.top_k, dim=-1)
        weight = torch.softmax(top_vals, dim=-1)

        output = torch.zeros_like(flat)
        #print(output.shape)
        for k in range(self.top_k):
            for e in range(len(self.experts)):
                mask = top_idx[:, k] == e
                # print(mask.shape, flat[mask].shape)
                if mask.any():
                    output[mask] += weight[mask, k:k+1] * self.experts[e](flat[mask])
        return output.reshape(x.shape)

In [31]:
# 🧪 Debug
moe = MixtureOfExperts(32, 64, num_experts=4, top_k=2)
x = torch.randn(2, 8, 32)
print('Output:', moe(x).shape)
print('Params:', sum(p.numel() for p in moe.parameters()))

Output: torch.Size([2, 8, 32])
Params: 16900


In [32]:
# ✅ SUBMIT
from torch_judge import check
check('moe')


🧪 Testing: Mixture of Experts (MoE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (8.0ms)
  ✅ [2/4] Has router and experts (1.6ms)
  ✅ [3/4] Router logits shape (9.6ms)
  ✅ [4/4] Gradient flow (14.8ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (33.9ms total)
  Progress saved. Run status() to see your dashboard.

